## Zigzag Conversionnies
The string "PAYPALISHIRING" is written in a zigzag pattern on a given number of rows like this: (you may want to display this pattern in a fixed font for better legibility)

P   A <br>  H   N
A P L <br> S I I G
 <br>Y   I   R
And then read line by line: "PAHNAPLSIIGYIR"

Write the code that will take a string and make this conversion given a number of rows:

string convert(string s, int numRows);
 

# Conventional Solution
The simulation approach solves the problem by mimicking the exact process of writing the string in a zigzag pattern row by row.

We create a separate container for each row and iterate through the characters of the input string one at a time. As we process each character, we place it into the current row. We then move either downward or upward through the rows depending on our current direction.

In [ ]:
class Solution:
    def convert(self, s: str, numRows: int) -> str:
        if numRows == 1 or numRows >= len(s):
            return s


        rows = [""] * numRows


        current_row = 0
        going_down = False

        for char in s:

            rows[current_row] += char

            if current_row == 0 or current_row == numRows - 1:
                going_down = not going_down

            if going_down:
                current_row += 1
            else:
                current_row -= 1

        
        return "".join(rows)

## Optimised Mathematical Approach

The optimized approach avoids explicitly simulating the zigzag movement. Instead, it relies on recognizing a repeating mathematical pattern in the character indices.

The key observation is that the zigzag structure repeats in cycles. For a zigzag with numRows, one complete cycle contains:

cycle_len = 2 * numRows - 2

characters.

Using this cycle length, we can directly determine which characters belong to each row without constructing the zigzag visually.

The first row contains characters spaced exactly one cycle apart.
The last row also follows the same rule.
The middle rows are special because each cycle contributes two characters:
1. a vertical character
2. a diagonal character

For each row, we iterate through the string in jumps of cycle_len. For middle rows, we additionally calculate the diagonal character index using:

diagonal_index = current_index + cycle_len - 2 * row

This allows us to build the final result directly in row order.

In [11]:
class Solution:
    def convert(self, s: str, numRows: int) -> str:
        if numRows == 1 or numRows >= len(s):
            return s

        result = []


        cycle_len = 2 * numRows - 2


        for row in range(numRows):

            for j in range(row, len(s), cycle_len):
                result.append(s[j])

                diagonal_index = j + cycle_len - 2 * row
                if (
                    row != 0
                    and row != numRows - 1
                    and diagonal_index < len(s)
                ):
                    result.append(s[diagonal_index])

        return "".join(result)

## Reverse unsigned integers

Given a signed 32-bit integer x, return x with its digits reversed. If reversing x causes the value to go outside the signed 32-bit integer range [-2<sup>31</sup>, 2<sup>31</sup> - 1], then return 0.

Assume the environment does not allow you to store 64-bit integers (signed or unsigned).

## Approach 1: Mathematical Digit Extraction
Instead of forming the full reversed number and checking afterwards, we build it one digit at a time using modulo and floor division — and critically, we check for overflow before each multiplication step. This respects the constraint of never actually holding a value outside the 32-bit range.
The key insight is that rev * 10 + digit > INT_MAX can be rearranged to rev > (INT_MAX - digit) // 10, which lets us detect the violation using only values already within bounds. 

Working on the absolute value throughout and reapply the sign at the end, this simplifies the overflow logic to a single direction (against INT_MAX), which is safe because |INT_MIN| > INT_MAX, so anything fitting in INT_MAX will always fit in the negative range too.

In [ ]:
class Solution:
    def reverse(self, x: int) -> int:
        INT_MAX = 2**31 - 1

        sign = -1 if x < 0 else 1
        x = abs(x)
        rev = 0

        while x:
            digit = x % 10
            x //= 10

            if rev > (INT_MAX - digit) // 10:
                return 0

            rev = rev * 10 + digit

        return sign * rev

## Approach 2: String Manipulation

The mental model: strip the sign, reverse the digit characters as a string, reattach the sign, then do a bounds check at the end. It's intuitive but leans on Python's arbitrary-precision integers under the hood.

In [ ]:
class Solution:
    def reverse(self, x: int) -> int:
        INT_MIN, INT_MAX = -(2**31), 2**31 - 1

        sign = -1 if x < 0 else 1
        result = sign * int(str(abs(x))[::-1])

        return result if INT_MIN <= result <= INT_MAX else 0

## Next Permutation
A permutation of an array of integers is an arrangement of its members into a sequence or linear order.

For example, for arr = [1,2,3], the following are all the permutations of arr: [1,2,3], [1,3,2], [2, 1, 3], [2, 3, 1], [3,1,2], [3,2,1].
The next permutation of an array of integers is the next lexicographically greater permutation of its integer. More formally, if all the permutations of the array are sorted in one container according to their lexicographical order, then the next permutation of that array is the permutation that follows it in the sorted container. If such arrangement is not possible, the array must be rearranged as the lowest possible order (i.e., sorted in ascending order).

For example, the next permutation of arr = [1,2,3] is [1,3,2].
Similarly, the next permutation of arr = [2,3,1] is [3,1,2].
While the next permutation of arr = [3,2,1] is [1,2,3] because [3,2,1] does not have a lexicographical larger rearrangement.
Given an array of integers nums, find the next permutation of nums.

The replacement must be in place and use only constant extra memory.

## Conventional Approach — Sort the Suffix
The idea here follows the same core logic as the optimised version, but takes a shortcut on the last step. After finding where the array "breaks" and doing the swap, we just sort the remaining suffix rather than thinking about why a reverse would suffice. It works, but it throws away information we actually already have, costing us O(n log n) on that final step.
Start from the right and find the first index i where the value dips below its right neighbour. That's the pivot. Then find the smallest value to the right of i that's still larger than nums[i], swap them, and sort everything to the right of i in ascending order. If no pivot exists, the whole array is in descending order, so just sort the entire thing.

In [ ]:
class Solution:
    def nextPermutation(self, nums: list[int]) -> None:
        n = len(nums)
        pivot = -1

        for i in range(n - 2, -1, -1):
            if nums[i] < nums[i + 1]:
                pivot = i
                break

        if pivot == -1:
            nums.sort()
            return

        for j in range(n - 1, pivot, -1):
            if nums[j] > nums[pivot]:
                nums[pivot], nums[j] = nums[j], nums[pivot]
                break

        nums[pivot + 1:] = sorted(nums[pivot + 1:])

## Optimised Approach — Reverse the Suffix
The key realisation is that once the pivod is found and the swap done, everything to the right of the pivot is guaranteed to still be in descending order. So instead of sorting it, you just reverse it to get ascending order in O(n). That one observation is what separates this solution from the conventional one.


Walk right to left to find the first index i where the sequence stops being non-increasing. That's your pivot, and nums[i] is the digit that needs to get bigger. Then walk right to left again and find the first value larger than nums[i], swap them. Now the suffix is still descending (swapping with the rightmost valid candidate preserves this), so a simple two-pointer reverse brings it to ascending order and gives the lexicographically smallest continuation.

In [ ]:
class Solution:
    def nextPermutation(self, nums: list[int]) -> None:
        n = len(nums)
        pivot = -1

        for i in range(n - 2, -1, -1):
            if nums[i] < nums[i + 1]:
                pivot = i
                break

        if pivot == -1:
            nums.reverse()
            return

        for j in range(n - 1, pivot, -1):
            if nums[j] > nums[pivot]:
                nums[pivot], nums[j] = nums[j], nums[pivot]
                break

        left, right = pivot + 1, n - 1
        while left < right:
            nums[left], nums[right] = nums[right], nums[left]
            left += 1
            right -= 1